# Char vs Word: RNN vs Transformer Language Models

**Assignment:** Architectural Deep Dive — Transformers vs. RNNs  
**Goal:** Build, train, and benchmark four models (CharRNN, CharTransformer, WordRNN, WordTransformer) on a Wikipedia corpus (~55k characters).

Pipeline:
1. Setup
2. Corpus Acquisition
3. Tokenization Pipelines
4. Model Definitions
5. Training (loss + perplexity)
6. Inference & Temperature Scaling
7. Export Results (`results/pic`, `results/data`)


## 1. Setup

Import libraries, set random seed, and select device (`cuda` if available, else `cpu`).


In [ ]:
# 导入实验所需的标准库与第三方库
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import requests
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

print("Imports OK")


In [ ]:
# 固定随机种子，保证实验可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 自动选择 GPU（云端 Colab 通常有 cuda）或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")


In [ ]:
# 创建输出目录：图表 -> results/pic，关键数值 -> results/data
# 云端运行时会在当前工作目录下生成这两个文件夹
PIC_DIR = Path("results/pic")
DATA_DIR = Path("results/data")
PIC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Output dirs:", PIC_DIR.resolve(), DATA_DIR.resolve())


## 2. Corpus Acquisition

**What:** Fetch plain text from Wikipedia API and clean it.  
**Input:** Article title(s), target length ≈ 55,000 characters.  
**Output:** Cleaned `raw_text` string.


In [ ]:
# Fetch one Wikipedia article as plain text


In [ ]:
def get_wikipedia_text(title="Germany"):
    """通过 Wikipedia API 拉取一篇英文维基页面的纯文本并做基础清洗。"""
    url = (
        "https://en.wikipedia.org/w/api.php"
        f"?action=query&prop=extracts&explaintext=1&titles={title}&format=json"
    )
    # Wikipedia 要求带 User-Agent，否则请求会被拒绝
    headers = {
        "User-Agent": "CharVSWordGPT/1.0 (Educational assignment; cloud notebook)"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    pages = response.json()["query"]["pages"]
    extract = list(pages.values())[0].get("extract", "")

    # 去掉 == Section == 这类标题标记，并把换行压成空格
    extract = re.sub(r"==+.*?==+", " ", extract)
    extract = re.sub(r"\s+", " ", extract).strip()
    return extract


In [ ]:
# Build a longer corpus by concatenating articles if needed


In [ ]:
def build_corpus(titles=None, min_chars=55000):
    """按需拼接多篇维基文章，直到达到约 55,000 字符。"""
    if titles is None:
        # 默认主题列表，可按需要增删
        titles = [
            "Germany",
            "Berlin",
            "European Union",
            "World War II",
            "German language",
        ]
    parts = []
    for title in titles:
        text = get_wikipedia_text(title)
        parts.append(text)
        total = sum(len(p) for p in parts)
        print(f"  + {title}: {len(text)} chars | cumulative={total}")
        if total >= min_chars:
            break
    return " ".join(parts)


In [ ]:
# 拉取并清洗语料（目标约 55,000 characters）
raw_text = build_corpus(min_chars=55000)
print(f"Corpus length: {len(raw_text)} characters")
print("Preview:", raw_text[:300], "...")


In [ ]:
# 若单次拉取仍不足，再补几篇（可按需修改标题列表）
if len(raw_text) < 55000:
    extra = build_corpus(
        titles=["Munich", "Frankfurt", "Hamburg", "Cologne", "Dresden"],
        min_chars=55000 - len(raw_text),
    )
    raw_text = (raw_text + " " + extra).strip()
    print(f"Augmented corpus length: {len(raw_text)}")

assert len(raw_text) >= 50000, "Corpus too short; add more Wikipedia titles."
print("Corpus ready.")


## 3. Tokenization Pipelines

Two parallel pipelines:

| Pipeline | Token unit | Notes |
|----------|------------|-------|
| Character-level | each char | small vocab, long sequences |
| Word-level | word / punctuation | frequency-filtered vocabulary |

**Output:** `char_encoded`, `word_encoded`, and lookup dicts.


In [ ]:
# --- PIPELINE A: Character-level tokenization ---
# 每个字符映射为一个整数索引
char_vocab = sorted(list(set(raw_text)))
char_to_int = {ch: i for i, ch in enumerate(char_vocab)}
int_to_char = {i: ch for i, ch in enumerate(char_vocab)}
char_encoded = np.array([char_to_int[ch] for ch in raw_text], dtype=np.int64)

print(f"Char vocab size: {len(char_vocab)}")
print(f"Char sequence length: {len(char_encoded)}")
print("Sample chars:", char_vocab[:20])


In [ ]:
# --- PIPELINE B: Word-level tokenization with frequency filter ---
# 用正则拆成“单词”或“标点”，再按词频过滤稀有词

MIN_WORD_FREQ = 2  # 出现次数 < 2 的词并入 <UNK>

word_tokens_raw = re.findall(r"\w+|[^\w\s]", raw_text, re.UNICODE)
freq = Counter(word_tokens_raw)

# 保留高频词；低频词用特殊符号 <UNK> 替代
kept_words = sorted([w for w, c in freq.items() if c >= MIN_WORD_FREQ])
word_vocab = ["<UNK>"] + kept_words
word_to_int = {w: i for i, w in enumerate(word_vocab)}
int_to_word = {i: w for i, w in enumerate(word_vocab)}

def word_to_id(w):
    return word_to_int.get(w, word_to_int["<UNK>"])

word_encoded = np.array([word_to_id(w) for w in word_tokens_raw], dtype=np.int64)

print(f"Raw word tokens: {len(word_tokens_raw)}")
print(f"Unique raw types: {len(freq)}")
print(f"Filtered word vocab size: {len(word_vocab)} (min_freq={MIN_WORD_FREQ})")
print(f"Word sequence length: {len(word_encoded)}")
print(f"UNK rate: {(word_encoded == 0).mean():.2%}")


In [ ]:
def generate_batches(data, batch_size, seq_length):
    """把一维 token 序列切成 (x, y) batch；y 是向右平移 1 位的下一个 token。"""
    total = batch_size * seq_length
    n_batches = len(data) // total
    if n_batches == 0:
        raise ValueError("Not enough data for the chosen batch_size/seq_length.")

    arr = data[: n_batches * total].reshape((batch_size, -1))
    # 注意：最后一段不够 seq_length+1 时停止，避免越界
    for n in range(0, arr.shape[1] - seq_length, seq_length):
        x = arr[:, n : n + seq_length]
        y = arr[:, n + 1 : n + seq_length + 1]
        yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


# 快速检查 batch 形状
_xb, _yb = next(generate_batches(char_encoded, batch_size=8, seq_length=32))
print("Batch x:", tuple(_xb.shape), "Batch y:", tuple(_yb.shape))


## 4. Model Definitions

Four variants share two architectures:

- **RNN**: embedding → multi-layer RNN → linear vocab projection
- **Transformer**: embedding + positional encoding → causal TransformerEncoder → linear projection


In [ ]:
class CharRNN(nn.Module):
    """字符/词级通用的 RNN 语言模型（作业中的 CharRNN / WordRNN）。"""

    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        # x: [batch, seq] -> embeds: [batch, seq, emb]
        embeds = self.embedding(x)
        out, hidden = self.rnn(embeds, hidden)
        # 展平时间维，送入全连接得到每个位置的词表 logits
        logits = self.fc(out.reshape(-1, self.hidden_size))
        return logits, hidden

    def init_hidden(self, batch_size):
        # 隐状态形状: [num_layers, batch, hidden]
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)


print(CharRNN)


In [ ]:
class CharTransformer(nn.Module):
    """带因果掩码的 Transformer 语言模型（CharTransformer / WordTransformer）。"""

    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 可学习位置编码（足够覆盖训练/生成时的最大序列长度）
        self.pos_encoder = nn.Parameter(torch.zeros(1, max_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        # 因果掩码：位置 i 只能看见 <= i 的 token（自回归生成）
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        src = self.embedding(x) + self.pos_encoder[:, :seq_len, :]
        out = self.transformer(src, mask=mask, is_causal=True)
        logits = self.out_fc(out.reshape(-1, self.d_model))
        return logits


print(CharTransformer)


In [ ]:
# 别名：同一套架构分别用于字符级与词级
WordRNN = CharRNN
WordTransformer = CharTransformer
print("Aliases ready: WordRNN, WordTransformer")


## 5. Training

Train all four models. For each epoch record:

- **mean cross-entropy loss**
- **perplexity** = `exp(loss)`

Hyperparameters are modest for cloud demo; increase epochs on Colab GPU if needed.


In [ ]:
# Training hyperparameters


In [ ]:
# 训练超参数（云端可酌情加大 epochs）
EPOCHS = 8
BATCH_SIZE = 32
CHAR_SEQ_LEN = 100
WORD_SEQ_LEN = 40
LR = 0.003
print("Hyperparameters ready:", {"EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE, "LR": LR})


In [ ]:
# Training loop: records loss and perplexity each epoch


In [ ]:
def train_model(model, data, mode="rnn", epochs=EPOCHS, batch_size=BATCH_SIZE, seq_length=40, name="model"):
    """训练单个模型，返回每个 epoch 的 loss 与 perplexity 列表。"""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    loss_history = []
    ppl_history = []

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        hidden = model.init_hidden(batch_size) if mode == "rnn" else None

        for x, y in generate_batches(data, batch_size=batch_size, seq_length=seq_length):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            if mode == "rnn":
                # 截断 BPTT：防止跨 batch 反传造成图爆炸
                hidden = hidden.detach()
                logits, hidden = model(x, hidden)
            else:
                logits = model(x)

            loss = criterion(logits, y.reshape(-1))
            loss.backward()
            # 梯度裁剪，缓解 RNN 梯度爆炸
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        mean_loss = total_loss / max(n_batches, 1)
        # 困惑度：对交叉熵取指数；数值越大表示越不确定
        ppl = math.exp(min(mean_loss, 20))  # 上限防止 overflow
        loss_history.append(mean_loss)
        ppl_history.append(ppl)
        print(f"[{name}] Epoch {epoch+1}/{epochs} | loss={mean_loss:.4f} | ppl={ppl:.2f}")

    return {"loss": loss_history, "ppl": ppl_history}


In [ ]:
# 初始化四个模型并放到 device
char_rnn = CharRNN(len(char_vocab)).to(device)
char_transformer = CharTransformer(len(char_vocab)).to(device)
word_rnn = WordRNN(len(word_vocab)).to(device)
word_transformer = WordTransformer(len(word_vocab)).to(device)

print("Char vocab:", len(char_vocab), "| Word vocab:", len(word_vocab))
print("Models initialized on", device)


In [ ]:
# Train 1/4: CharRNN
print("Training CharRNN...")
hist_char_rnn = train_model(
    char_rnn, char_encoded, mode="rnn",
    seq_length=CHAR_SEQ_LEN, name="CharRNN"
)


In [ ]:
# Train 2/4: CharTransformer
print("Training CharTransformer...")
hist_char_transformer = train_model(
    char_transformer, char_encoded, mode="transformer",
    seq_length=CHAR_SEQ_LEN, name="CharTransformer"
)


In [ ]:
# Train 3/4: WordRNN
print("Training WordRNN...")
hist_word_rnn = train_model(
    word_rnn, word_encoded, mode="rnn",
    seq_length=WORD_SEQ_LEN, name="WordRNN"
)


In [ ]:
# Train 4/4: WordTransformer
print("Training WordTransformer...")
hist_word_transformer = train_model(
    word_transformer, word_encoded, mode="transformer",
    seq_length=WORD_SEQ_LEN, name="WordTransformer"
)
print("All four models trained.")


### 5.1 Convergence Plots

**What:** Compare training loss and perplexity across the four models.  
**X-axis:** Epoch (1 … N), unitless count.  
**Y-axis:** Cross-entropy loss (nats) / Perplexity (unitless, lower is better).


In [ ]:
# 汇总历史，便于画图与导出
histories = {
    "CharRNN": hist_char_rnn,
    "CharTransformer": hist_char_transformer,
    "WordRNN": hist_word_rnn,
    "WordTransformer": hist_word_transformer,
}

epochs_axis = list(range(1, EPOCHS + 1))

plt.figure(figsize=(10, 5))
styles = {
    "CharRNN": ("--", "o"),
    "CharTransformer": ("--", "x"),
    "WordRNN": ("-", "o"),
    "WordTransformer": ("-", "x"),
}
for name, hist in histories.items():
    ls, mk = styles[name]
    plt.plot(epochs_axis, hist["loss"], linestyle=ls, marker=mk, label=name)

plt.xlabel("Epoch (count)")
plt.ylabel("Mean cross-entropy loss (nats)")
plt.title("Training Loss vs Epoch")
plt.xlim(1, EPOCHS)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
loss_fig_path = PIC_DIR / "fig_training_loss.png"
plt.savefig(loss_fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", loss_fig_path)


In [ ]:
plt.figure(figsize=(10, 5))
for name, hist in histories.items():
    ls, mk = styles[name]
    plt.plot(epochs_axis, hist["ppl"], linestyle=ls, marker=mk, label=name)

plt.xlabel("Epoch (count)")
plt.ylabel("Perplexity = exp(loss) (unitless, lower=better)")
plt.title("Training Perplexity vs Epoch")
plt.xlim(1, EPOCHS)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
ppl_fig_path = PIC_DIR / "fig_training_perplexity.png"
plt.savefig(ppl_fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", ppl_fig_path)


In [ ]:
# 最终 epoch 指标表（写论文用）
final_metrics = {
    name: {
        "final_loss": hist["loss"][-1],
        "final_ppl": hist["ppl"][-1],
        "best_ppl": min(hist["ppl"]),
    }
    for name, hist in histories.items()
}
print(json.dumps(final_metrics, indent=2))


## 6. Inference & Coherence

**What:** Generate text from a fixed prompt for all four models; then vary temperature.  
**Input:** prompt string + trained models.  
**Output:** generated samples for qualitative comparison.


In [ ]:
# Character-level text generation


In [ ]:
def generate_char_level(model, prompt, length=200, mode="rnn", temperature=1.0):
    """字符级自回归生成；temperature 控制采样尖锐程度。"""
    model.eval()
    chars = list(prompt)
    hidden = model.init_hidden(1) if mode == "rnn" else None

    for _ in range(length):
        context = chars[-CHAR_SEQ_LEN:]
        x = torch.tensor([[char_to_int.get(c, 0) for c in context]], dtype=torch.long, device=device)
        with torch.no_grad():
            if mode == "rnn":
                out, hidden = model(x, hidden)
                logits = out[-1]
            else:
                logits = model(x)[-1]
        # temperature 越小越“确定”，越大越随机
        probs = F.softmax(logits / max(temperature, 1e-6), dim=0).cpu().numpy()
        next_idx = np.random.choice(len(char_vocab), p=probs)
        chars.append(int_to_char[next_idx])
    return "".join(chars)


In [ ]:
# Word-level text generation


In [ ]:
def generate_word_level(model, prompt, length=40, mode="rnn", temperature=1.0):
    """词级自回归生成；分词方式必须与训练时一致。"""
    model.eval()
    words = re.findall(r"\w+|[^\w\s]", prompt, re.UNICODE)
    if not words:
        words = ["Germany"]
    hidden = model.init_hidden(1) if mode == "rnn" else None

    for _ in range(length):
        context = words[-WORD_SEQ_LEN:]
        x = torch.tensor([[word_to_id(w) for w in context]], dtype=torch.long, device=device)
        with torch.no_grad():
            if mode == "rnn":
                out, hidden = model(x, hidden)
                logits = out[-1]
            else:
                logits = model(x)[-1]
        probs = F.softmax(logits / max(temperature, 1e-6), dim=0).cpu().numpy()
        next_idx = np.random.choice(len(word_vocab), p=probs)
        words.append(int_to_word[next_idx])

    # 简单拼接：标点不加前导空格
    out = ""
    for w in words:
        out += w if re.match(r"[^\w\s]", w) else (" " + w)
    return out.strip()


In [ ]:
# 固定提示文本（四个模型用同一 prompt，便于横向对比）
PROMPT = "Germany is a"

print("=" * 60)
print("PROMPT:", PROMPT)
print("=" * 60)

gen_char_rnn = generate_char_level(char_rnn, PROMPT, length=200, mode="rnn")
print("\n--- CharRNN ---\n", gen_char_rnn)

gen_char_trans = generate_char_level(char_transformer, PROMPT, length=200, mode="transformer")
print("\n--- CharTransformer ---\n", gen_char_trans)


In [ ]:
gen_word_rnn = generate_word_level(word_rnn, PROMPT, length=40, mode="rnn")
print("--- WordRNN ---\n", gen_word_rnn)

gen_word_trans = generate_word_level(word_transformer, PROMPT, length=40, mode="transformer")
print("\n--- WordTransformer ---\n", gen_word_trans)


### 6.1 Temperature Scaling

Lower temperature → sharper distribution → more conservative / repetitive text.  
Higher temperature → flatter distribution → more diverse / chaotic text.


In [ ]:
# 仅用 WordTransformer 演示温度效应（也可改成其他模型）
temps = [0.2, 0.7, 1.0, 1.5, 2.0]
temp_samples = {}
for t in temps:
    sample = generate_word_level(
        word_transformer, PROMPT, length=40, mode="transformer", temperature=t
    )
    temp_samples[str(t)] = sample
    print(f"\n[temperature={t}]\n{sample}")


In [ ]:
# 第二个 prompt，补充生成样例
PROMPT2 = "BMW is a car"
print("PROMPT2:", PROMPT2)
print("CharTransformer:", generate_char_level(char_transformer, PROMPT2, 120, "transformer")[:300])
print("WordTransformer:", generate_word_level(word_transformer, PROMPT2, 30, "transformer"))


## 7. Export Results

Save figures to `results/pic/` and key metrics / generated samples to `results/data/` for the paper write-up.


In [ ]:
# 导出训练曲线原始数据（每个 epoch 的 loss / ppl）
curves_payload = {
    "epochs": epochs_axis,
    "models": {
        name: {"loss": hist["loss"], "perplexity": hist["ppl"]}
        for name, hist in histories.items()
    },
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "char_seq_len": CHAR_SEQ_LEN,
        "word_seq_len": WORD_SEQ_LEN,
        "lr": LR,
        "min_word_freq": MIN_WORD_FREQ,
        "seed": SEED,
        "device": str(device),
    },
    "corpus": {
        "num_characters": len(raw_text),
        "char_vocab_size": len(char_vocab),
        "word_vocab_size": len(word_vocab),
        "num_word_tokens": int(len(word_encoded)),
    },
}
curves_path = DATA_DIR / "training_curves.json"
with open(curves_path, "w", encoding="utf-8") as f:
    json.dump(curves_payload, f, ensure_ascii=False, indent=2)
print("Saved:", curves_path)


In [ ]:
# 导出最终指标表
metrics_path = DATA_DIR / "final_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)
print("Saved:", metrics_path)


In [ ]:
# 导出生成样例（固定 prompt + 温度实验）
generation_payload = {
    "prompt": PROMPT,
    "samples": {
        "CharRNN": gen_char_rnn,
        "CharTransformer": gen_char_trans,
        "WordRNN": gen_word_rnn,
        "WordTransformer": gen_word_trans,
    },
    "temperature_samples_WordTransformer": temp_samples,
}
gen_path = DATA_DIR / "generation_samples.json"
with open(gen_path, "w", encoding="utf-8") as f:
    json.dump(generation_payload, f, ensure_ascii=False, indent=2)
print("Saved:", gen_path)


In [ ]:
# 再存一份便于粘贴到论文的 Markdown 结果摘要
md_lines = [
    "# Experiment Results Summary",
    "",
    f"- Corpus characters: {len(raw_text)}",
    f"- Char vocab size: {len(char_vocab)}",
    f"- Word vocab size: {len(word_vocab)} (min_freq={MIN_WORD_FREQ})",
    f"- Device: {device}",
    f"- Epochs: {EPOCHS}",
    "",
    "## Final Metrics",
    "",
    "| Model | Final Loss | Final PPL | Best PPL |",
    "|---|---:|---:|---:|",
]
for name, m in final_metrics.items():
    md_lines.append(
        f"| {name} | {m['final_loss']:.4f} | {m['final_ppl']:.2f} | {m['best_ppl']:.2f} |"
    )
md_lines += [
    "",
    "## Figures",
    "",
    "- ![Training loss](../pic/fig_training_loss.png)",
    "- ![Training perplexity](../pic/fig_training_perplexity.png)",
    "",
    "## Notes for paper",
    "",
    "- Character-level loss is often lower because the vocab is much smaller (~100 vs thousands).",
    "- Compare spelling quality (char models) vs grammatical coherence (word models).",
    "- Discuss temperature effects on diversity vs fluency.",
    "",
]
summary_path = DATA_DIR / "results_summary.md"
summary_path.write_text("\n".join(md_lines), encoding="utf-8")
print("Saved:", summary_path)
print("\nAll exports done.")
print("pics:", list(PIC_DIR.glob('*')))
print("data:", list(DATA_DIR.glob('*')))


## Done

Next steps for the paper:
1. Put the two figures into the Analysis section with captions (Figure 1 / Figure 2).
2. Quote final PPL numbers from `results/data/final_metrics.json`.
3. Paste short generation samples and discuss coherence / temperature.
